In [1]:
from quam_libs.quantum_memory.legacy.NoiseAnalyze import QuantumMemory

/Users/jackchao/Desktop/Project/publication/quantum_memory/data_availability/quantum_memory_data/.venv/lib/python3.12/site-packages/pennylane/__init__.py:212: PennyLaneDeprecationWarning: PennyLane v0.44 has dropped maintainence support for NumPy < 2.0.0. You have version 1.26.4 installed. Future versions of PennyLane will not work with NumPy<2.0. Please consider upgrading NumPy using `python -m pip install numpy --upgrade`. 
  warnings.warn(


In [2]:
import numpy as np
axes = np.array([0.99975927, 1.00011047, 0.99992695])
center= np.array([ 1.24212753e-04, -1.68326924e-05,  1.16281414e-04])
R = np.array([[-0.64153201,  0.09993454, -0.76055885],
              [-0.70843317,  0.30308275,  0.63738787],
              [-0.29420933, -0.94770984,  0.12364034]])
param = np.array([ 4.99999112e-01,  5.00006982e-01,  5.00044649e-01,  5.72845696e-05,
  3.29392913e-04, -9.63899270e-05, -1.24249871e-04,  1.68370203e-05,
 -1.16334335e-04, -4.99949101e-01])

In [3]:
from quam_libs.analyzer import QuantumMemoryAnalyze
import itertools
# Compute Choi state from ellipsoid parameters


perm_robustness_dict = {}
for i in range(1):
    print(f"Results for:")
    print(f"R :\n{R}")
    print(f"Center :\n{center}")
    print(f"Axes :\n{axes}")
    
    # Compute Choi state from ellipsoid parameters
    choi = QuantumMemoryAnalyze.choi_state(center, axes, R)
    correct_choi = QuantumMemoryAnalyze.correct_choi(choi)
    # Recompute robustness from the Choi state using static method
    robustness = QuantumMemoryAnalyze.memory_robustness(correct_choi)
    print(f"Quantum robustness for: {robustness:.3f}")
    print('-'*75)

    perms = list(itertools.permutations([0,1,2]))
    perm_robustness_dict = {}
    for perm in np.array(perms):
        # Properly permute rotation matrix columns to match axes permutation
        R_perm = R[:, perm]
        axes_perm = axes[perm]
        
        # Compute robustness for this permutation
        choi_perm = QuantumMemoryAnalyze.choi_state(center, axes_perm, R_perm)
        correct_choi_perm = QuantumMemoryAnalyze.correct_choi(choi_perm)
        robustness_perm = QuantumMemoryAnalyze.memory_robustness(correct_choi_perm)
        
        perm_robustness_dict[tuple(perm)] = robustness_perm
        print(f"Permutation {perm} robustness: {robustness_perm:.3f}")
    
    # Find permutation with maximum robustness
    max_perm = list(max(perm_robustness_dict, key=perm_robustness_dict.get))
    print(f"Best permutation {max_perm} with robustness {perm_robustness_dict[tuple(max_perm)]:.3f}")

    # Apply best permutation to rotation matrix columns
    R = R[:, max_perm]
    axes = axes[max_perm]
    if np.linalg.det(R) < 0:
        R[:,2]*=-1 
    print(f'after swap')
    print(f"R :\n{R} and det(R) = {np.linalg.det(R)}")
    print(f"Center :\n{center}")
    print(f"Axes :\n{axes}")
    choi = QuantumMemoryAnalyze.choi_state(center, axes, R)
    correct_choi = QuantumMemoryAnalyze.correct_choi(choi)
    robustness = QuantumMemoryAnalyze.memory_robustness(correct_choi)
    print(f"Robustness for: {robustness:.3f}")
    print(f"Choi state for:\n{correct_choi}")
    print('-'*100)

Results for:
R :
[[-0.64153201  0.09993454 -0.76055885]
 [-0.70843317  0.30308275  0.63738787]
 [-0.29420933 -0.94770984  0.12364034]]
Center :
[ 1.24212753e-04 -1.68326924e-05  1.16281414e-04]
Axes :
[0.99975927 1.00011047 0.99992695]
After 2 iterations, the Choi state is valid.
Quantum robustness for: 0.000
---------------------------------------------------------------------------
After 2 iterations, the Choi state is valid.
Permutation [0 1 2] robustness: 0.000
After 1 iterations, the Choi state is valid.
Permutation [0 2 1] robustness: 0.976
After 1 iterations, the Choi state is valid.
Permutation [1 0 2] robustness: 0.742
After 2 iterations, the Choi state is valid.
Permutation [1 2 0] robustness: 0.000
After 2 iterations, the Choi state is valid.
Permutation [2 0 1] robustness: 0.000
After 1 iterations, the Choi state is valid.
Permutation [2 1 0] robustness: 0.876
Best permutation [0, 2, 1] with robustness 0.976
after swap
R :
[[-0.64153201 -0.76055885  0.09993454]
 [-0.7084331